# Galaxy N-Body: run the CUDA code on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The notebook builds the project, runs the GPU benchmarks and the block-size sweep, tries Nsight Compute, regenerates the charts and README results section, and renders a galaxy animation.
3. At the end, download `results.zip` and commit `results/` + `README.md` to the repo (or push from the notebook with a token).

The full GPU section takes about 5–10 minutes on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
!nvcc --version | tail -2
!cmake --version | head -1

## Get the code
If the repository is private, use `https://<TOKEN>@github.com/...` with a GitHub personal access token.

In [ ]:
REPO = "https://github.com/colmeabhi/parallelized-nbody-simulation-cuda-cpp.git"
BRANCH = "claude/fervent-planck-p0ob3o"   # change to "main" once merged
!rm -rf nbody-repo && git clone -q --branch $BRANCH $REPO nbody-repo
%cd nbody-repo
!git log --oneline -3

## Build
`CUDA_ARCH` is read from the GPU so CMake does not need `native` detection.

In [ ]:
import subprocess
cc = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
CUDA_ARCH = cc.replace(".", "")
print("CUDA_ARCH =", CUDA_ARCH)
!cmake -S nbody -B nbody/build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH > /dev/null && cmake --build nbody/build -j 4 2>&1 | tail -3

## Correctness smoke test
A short KDK run: the energy drift should be small, and `tiled` and `naive` should agree.

In [ ]:
!./nbody/build/nbody_bench_gpu --header --kernel tiled --n 4096 --steps 200 --dt 0.0025 --energy
!./nbody/build/nbody_bench_gpu          --kernel naive --n 4096 --steps 200 --dt 0.0025 --energy

## GPU benchmarks (scaling, block-size sweep, energy)
`SKIP_CPU=1` keeps the CPU numbers already committed (Colab's 2 vCPUs would make a weaker baseline). Remove it to re-measure the CPU here too.

In [ ]:
!SKIP_CPU=1 CUDA_ARCH=$CUDA_ARCH bash scripts/run_benchmarks.sh

## Nsight Compute (optional)
Colab often blocks GPU performance counters (`ERR_NVGPUCTRPERM`). If this cell fails, run `scripts/profile_ncu.sh` on a machine where you have admin access to the GPU.

In [ ]:
import os, shutil
ncu = shutil.which("ncu") or next((p for p in ["/usr/local/cuda/bin/ncu", "/opt/nvidia/nsight-compute/ncu"] if os.path.exists(p)), None)
print("ncu:", ncu)
if ncu:
    os.environ["PATH"] = os.path.dirname(ncu) + ":" + os.environ["PATH"]
    !bash scripts/profile_ncu.sh || echo "Nsight Compute could not read counters here"

## Report: charts + README results section

In [ ]:
!pip -q install matplotlib numpy
!python3 scripts/make_report.py
from IPython.display import Image, Markdown, display
for png in ["scaling.png", "block_sweep.png", "energy.png", "bh_crossover.png"]:
    if os.path.exists(f"results/{png}"):
        display(Image(f"results/{png}"))
display(Markdown(open("results/RESULTS.md").read()))

## Galaxy animation

In [ ]:
!cd nbody && ./build/galaxy_nbody 4096 3000 --dt 0.005 | tail -6
!cd nbody && python3 visualize.py simulation.bin galaxy.gif
display(Image("nbody/galaxy.gif"))

## Save the results
Download the zip and commit `results/` and `README.md`, or push directly (needs a token with write access).

In [ ]:
!zip -qr results.zip results README.md && ls -la results.zip
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    pass

In [ ]:
# Optional: push from Colab.
# !git config user.name "abhi" && git config user.email "ahirraoabhishek007@gmail.com"
# !git add results README.md && git commit -m "Add GPU benchmark results (Colab T4)"
# !git push https://<TOKEN>@github.com/colmeabhi/parallelized-nbody-simulation-cuda-cpp.git HEAD:$BRANCH